# Pi* — Reprise entraînement Deep RL (Kaggle)

**Checkpoint** : 1 000 000 steps déjà effectués  
**Restant** : 2 000 000 steps  

**Avant de lancer** : activer Internet + GPU T4 dans le panneau droit.

In [ ]:
# Cellule 1 — Installation
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'stable-baselines3>=2.3.0', 'sb3-contrib>=2.3.0',
                'gymnasium>=0.29.0', 'tqdm', 'rich'], check=True)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

In [ ]:
# Cellule 2 — Détection automatique des chemins
import os, zipfile, shutil
from pathlib import Path

print('Fichiers disponibles dans /kaggle/input :')
all_files = []
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        p = os.path.join(root, f)
        print(f'  {p}')
        all_files.append(p)

# Dossiers aussi (Kaggle dézippé → dossier)
all_dirs = []
for root, dirs, files in os.walk('/kaggle/input'):
    for d in dirs:
        all_dirs.append(os.path.join(root, d))

OUTPUT_DIR = Path('/kaggle/working')

def find_file(name):
    matches = [f for f in all_files if name in f]
    if not matches:
        raise FileNotFoundError(f'Fichier introuvable : {name}')
    return Path(matches[0])

PARQUET_PATH = find_file('btcusdt_1m.parquet')
VECNORM_PATH = find_file('deep_agent_vecnormalize_1000000_steps.pkl')

# Modèle : zip conservé ou dossier extrait par Kaggle
model_zip    = [f for f in all_files if 'deep_agent_1000000_steps.zip' in f]
model_folder = [d for d in all_dirs  if 'deep_agent_1000000_steps' in d]

dst_zip  = OUTPUT_DIR / 'deep_agent_1000000_steps.zip'
CKPT_MODEL = OUTPUT_DIR / 'deep_agent_1000000_steps'

if model_zip:
    shutil.copy(model_zip[0], dst_zip)
    print(f'Zip copié : {dst_zip}')
elif model_folder:
    # Re-zippe avec les fichiers à la RACINE du zip (requis par SB3)
    folder = Path(model_folder[0])
    with zipfile.ZipFile(dst_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in folder.iterdir():
            zf.write(f, f.name)
    print(f'Re-zippé (structure correcte) : {dst_zip}')
else:
    # Dernier recours : fichier sans extension .zip
    candidates = [f for f in all_files if 'deep_agent_1000000' in f and not f.endswith('.pkl')]
    if candidates:
        shutil.copy(candidates[0], dst_zip)
        print(f'Fichier copié : {dst_zip}')
    else:
        raise FileNotFoundError('Modèle deep_agent_1000000_steps introuvable')

print(f'\nParquet  : {PARQUET_PATH}')
print(f'VecNorm  : {VECNORM_PATH}')
print(f'Modèle   : {dst_zip}')

In [ ]:
# Cellule 3 — Feature Engineering + Environnement
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

N_FEATURES   = 9
_CLOSE_CLIP  = 0.03
_VOL_CLIP    = 5.0
_FUND_CLIP   = 0.005
FEATURE_COLS = ['f_close_pct','f_body_pct','f_upper_wick','f_lower_wick',
                'f_vol_ratio','f_delta','f_funding','f_session']

def aggregate_5m(df_1m):
    df = df_1m.copy()
    df['ts_5m'] = (df['ts'] // 300) * 300
    agg = df.groupby('ts_5m').agg(
        open=('open','first'), high=('high','max'),
        low=('low','min'),     close=('close','last'),
        volume=('volume','sum'),
        taker_buy_vol=('taker_buy_vol','sum'),
        funding_rate=('funding_rate','last'),
    ).reset_index().rename(columns={'ts_5m':'ts'})
    agg['timestamp'] = pd.to_datetime(agg['ts'], unit='s', utc=True)
    return agg.sort_values('ts').reset_index(drop=True)

def label_session(ts_utc):
    hour = ts_utc.dt.hour
    s = pd.Series(0, index=ts_utc.index, dtype=int)
    s[(hour >= 7)  & (hour < 13)] = 1
    s[(hour >= 13) & (hour < 16)] = 2
    s[(hour >= 16) & (hour < 22)] = 3
    return s

def compute_features(df_1m):
    df = aggregate_5m(df_1m)
    df['session'] = label_session(df['timestamp'])
    close, open_, high, low, vol = df['close'], df['open'], df['high'], df['low'], df['volume']
    rng = (high - low).clip(lower=1e-9)
    df['f_close_pct']  = (close.pct_change().fillna(0).clip(-_CLOSE_CLIP,_CLOSE_CLIP) / _CLOSE_CLIP).astype('float32')
    df['f_body_pct']   = ((close - open_) / rng).clip(-1, 1).astype('float32')
    df['f_upper_wick'] = ((high - pd.concat([open_,close],axis=1).max(axis=1)) / rng).clip(0,1).astype('float32')
    df['f_lower_wick'] = ((pd.concat([open_,close],axis=1).min(axis=1) - low) / rng).clip(0,1).astype('float32')
    avg_vol = vol.rolling(20, min_periods=1).mean().clip(lower=1e-9)
    df['f_vol_ratio']  = ((vol / avg_vol).clip(0,_VOL_CLIP) / _VOL_CLIP).astype('float32')
    df['f_delta']      = (df['taker_buy_vol'] / vol.clip(lower=1e-9)).clip(0,1).astype('float32')
    df['f_funding']    = (df['funding_rate'].fillna(0).clip(-_FUND_CLIP,_FUND_CLIP) / _FUND_CLIP).astype('float32')
    df['f_session']    = (df['session'].astype(float) / 3.0).astype('float32')
    return df.reset_index(drop=True)

def obs_from_row(row, position):
    return np.concatenate([row[FEATURE_COLS].values.astype(np.float32),
                           np.array([float(position)], dtype=np.float32)])

FEE, SLIP, PENALTY = 0.0005, 0.0002, 1.5
_SL_LOW, _TP_LOW   = 0.003, 0.006
_SL_HIGH, _TP_HIGH = 0.004, 0.008
_SL_EXT, _TP_EXT   = 0.006, 0.012

def sl_tp(vol_ratio):
    if vol_ratio > 0.8:  return _SL_EXT,  _TP_EXT
    if vol_ratio > 0.5:  return _SL_HIGH, _TP_HIGH
    return _SL_LOW, _TP_LOW

class DeepTradingEnv(gym.Env):
    metadata = {'render_modes': []}
    def __init__(self, df_feat):
        super().__init__()
        self._sessions = self._split(df_feat)
        self._idx = 0
        self.observation_space = spaces.Box(-1., 2., shape=(N_FEATURES,), dtype=np.float32)
        self.action_space = spaces.Discrete(3)
        self._ep = self._pos = self._step = 0
        self._price = self._sl = self._tp = None
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._ep   = self._sessions[self._idx].reset_index(drop=True)
        self._idx  = (self._idx + 1) % len(self._sessions)
        self._step = 0; self._pos = 0; self._price = None
        self._sl, self._tp = _SL_LOW, _TP_LOW
        return self._obs(), {}
    def step(self, action):
        row   = self._ep.iloc[self._step]
        price = float(row['close'])
        pnl   = self._pnl(price)
        sl_hit = self._pos != 0 and (pnl <= -self._sl or pnl >= self._tp)
        if sl_hit: action = 0
        desired = {0:0, 1:1, 2:-1}[int(action)]
        closing = self._pos != 0 and (desired == 0 or desired != self._pos)
        reward  = (pnl * PENALTY if pnl < 0 else pnl) if (closing or sl_hit) else 0.0
        self._act(int(action), price, row)
        self._step += 1
        done = self._step >= len(self._ep)
        if done and self._pos != 0:
            lp = float(self._ep.iloc[-1]['close'])
            lpl = self._pnl(lp)
            reward += lpl * PENALTY if lpl < 0 else lpl
            self._pos = 0; self._price = None
        obs = self._obs() if not done else np.zeros(N_FEATURES, dtype=np.float32)
        return obs, float(reward), done, False, {}
    def render(self): pass
    @property
    def n_episodes(self): return len(self._sessions)
    def _obs(self):
        return obs_from_row(self._ep.iloc[min(self._step, len(self._ep)-1)], self._pos)
    def _pnl(self, price):
        if not self._pos or self._price is None: return 0.0
        return float(self._pos * (price - self._price) / self._price - FEE - SLIP)
    def _act(self, action, price, row):
        d = {0:0, 1:1, 2:-1}[action]
        if d == self._pos: return
        if self._pos: self._pos = 0; self._price = None
        if d:
            self._pos   = d
            self._price = price * (1 + d * SLIP)
            self._sl, self._tp = sl_tp(float(row.get('f_vol_ratio', 0.)))
    def _split(self, df):
        df = df.copy()
        df['_k'] = df['timestamp'].dt.date.astype(str) + '_' + df['session'].astype(str)
        return [g.drop(columns=['_k']) for _, g in df.groupby('_k', sort=True) if len(g) >= 6]

print('Chargement parquet...')
df_1m = pd.read_parquet(PARQUET_PATH)
print(f'  {len(df_1m):,} bougies 1m | {df_1m["timestamp"].iloc[0]} → {df_1m["timestamp"].iloc[-1]}')

print('Calcul features 5min...')
df_feat = compute_features(df_1m)
env_test = DeepTradingEnv(df_feat)
obs, _ = env_test.reset()
print(f'  {len(df_feat):,} bougies 5min | obs shape : {obs.shape} | Sessions : {env_test.n_episodes}')

In [ ]:
# Cellule 4 — Reprise entraînement
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.callbacks import CheckpointCallback
import torch

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
REMAINING = 2_000_000

print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

checkpoint_cb = CheckpointCallback(
    save_freq         = 200_000,
    save_path         = str(OUTPUT_DIR / 'checkpoints'),
    name_prefix       = 'deep_agent',
    save_vecnormalize = True,
)

vec_env = make_vec_env(lambda: DeepTradingEnv(df_feat), n_envs=1)
vec_env = VecNormalize.load(str(VECNORM_PATH), vec_env)
vec_env.training    = True
vec_env.norm_reward = True

model = RecurrentPPO.load(str(CKPT_MODEL), env=vec_env, device=DEVICE)
print(f'Checkpoint chargé — {model.num_timesteps:,} timesteps déjà effectués')
print(f'Reprise pour {REMAINING:,} timesteps supplémentaires...\n')

model.learn(
    total_timesteps     = REMAINING,
    callback            = checkpoint_cb,
    reset_num_timesteps = False,
    progress_bar        = True,
)

model.save(str(OUTPUT_DIR / 'deep_agent'))
vec_env.save(str(OUTPUT_DIR / 'deep_vecnorm.pkl'))
print(f'\nModèle final : /kaggle/working/deep_agent.zip')
print(f'VecNormalize : /kaggle/working/deep_vecnorm.pkl')
print('\n→ Téléchargez via Output (bouton en haut à droite) → placez dans db/')